In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from pyproj import datadir
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")


In [ ]:
output_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")
shapefile_path = Path("/home/jupyter-daniela/suyana/geometries/areas_pesca_peru/areas_pesca_peru.shp")

# --- Load individual calas with nearest-pixel env data ---
df_calas = pd.read_csv(output_path / "calas_with_hycom_data.csv", parse_dates=["date"])

# --- Assign DPTO via spatial join ---
gdf_peru = gpd.read_file(shapefile_path)[["DPTO", "geometry"]]
gdf_calas = gpd.GeoDataFrame(
    df_calas,
    geometry=gpd.points_from_xy(df_calas["lon"], df_calas["lat"]),
    crs="EPSG:4326"
)
gdf_joined = gpd.sjoin(gdf_calas, gdf_peru, how="inner", predicate="within")
df_calas = pd.DataFrame(gdf_joined.drop(columns=["geometry", "index_right"]))

# --- Map region ---
mapa_regiones = {
    "PIURA": "norte", "LAMBAYEQUE": "norte", "TUMBES": "norte",
    "LA LIBERTAD": "centro", "ANCASH": "centro", "LIMA": "centro", "ICA": "centro",
    "AREQUIPA": "sur", "MOQUEGUA": "sur", "TACNA": "sur"
}
df_calas["region"] = df_calas["DPTO"].map(mapa_regiones)
df_calas = df_calas.rename(columns={"DPTO": "dept"})

# --- Aggregate to daily/dept ---
df_merge = (
    df_calas
    .groupby(["date", "dept", "region", "season"])
    .agg(
        total_catch_tm  =("catch_tm",       "sum"),
        hycom_temp      =("hycom_temp",      "mean"),
        hycom_sal       =("hycom_sal",       "mean"),
        hycom_temp_anom =("hycom_temp_anom", "mean"),
        modis_sst       =("modis_sst",       "mean"),
        modis_sst_anom  =("modis_sst_anom",  "mean"),
        chl             =("chl",             "mean"),
    )
    .reset_index()
)
df_merge["date"] = pd.to_datetime(df_merge["date"]).dt.tz_localize(None)

# --- Salinity anomaly vs 35.1 PSU ---
df_merge["hycom_sal_anom_ref35"] = df_merge["hycom_sal"] - 35.1

# --- Salinity anomaly vs day-of-year climatology ---
df_merge["doy"] = df_merge["date"].dt.dayofyear
sal_clim = df_merge.groupby("doy")["hycom_sal"].mean()
df_merge["hycom_sal_anom"] = df_merge["hycom_sal"] - df_merge["doy"].map(sal_clim)

# --- CHL anomaly vs day-of-year climatology ---
chl_clim = df_merge.groupby("doy")["chl"].mean()
df_merge["chl_anom"] = df_merge["chl"] - df_merge["doy"].map(chl_clim)

# --- Log CHL and its anomaly vs day-of-year climatology ---
df_merge["log_chl"] = np.log(df_merge["chl"].clip(lower=1e-6))
log_chl_clim = df_merge.groupby("doy")["log_chl"].mean()
df_merge["log_chl_anom"] = df_merge["log_chl"] - df_merge["doy"].map(log_chl_clim)

df_merge = df_merge.drop(columns="doy")

print(df_merge.shape)
print(df_merge.columns.tolist())


In [6]:
df_merge

,fecha,total_pescado_tm,DPTO,temporada,region_macro,temperatura,salinidad,anom_temp,anom_sal,anom_sal_ref35
0,2016-06-18,2830.0,ANCASH,1ra 2016,centro,19.101004,35.064500,0.783262,0.031097,-0.035500
1,2016-06-18,2830.0,ANCASH,1ra 2016,centro,18.869705,35.043457,0.551964,0.010056,-0.056541
2,2016-06-18,2830.0,ANCASH,1ra 2016,centro,18.641376,35.065740,0.323635,0.032337,-0.034260
3,2016-06-18,2830.0,ANCASH,1ra 2016,centro,18.170180,35.107826,-0.941936,-0.034996,0.007828
4,2016-06-18,2830.0,ANCASH,1ra 2016,centro,19.226192,35.165535,0.114077,0.022713,0.065536
...,...,...,...,...,...,...,...,...,...,...
60175,2023-02-03,90.0,PIURA,2da 2022,norte,20.734848,35.118244,-1.432270,0.008141,0.018246
60176,2023-02-03,90.0,PIURA,2da 2022,norte,20.857475,35.035290,-1.309643,-0.074814,-0.064709
60177,2023-02-03,90.0,PIURA,2da 2022,norte,21.389566,34.877080,-2.302939,0.282509,-0.222919
60178,2023-02-03,90.0,PIURA,2da 2022,norte,24.138838,34.409252,0.446333,-0.185318,-0.690746


In [ ]:
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

df_model = df_merge.copy()
df_model = df_model.set_index(["region", "date"])

y = df_model["total_catch_tm"]
X = df_model[["hycom_temp_anom", "hycom_sal_anom_ref35"]]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res = mod.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


In [ ]:
df_model = df_merge.copy()
df_model = df_model.set_index(["region", "date"])
df_model = df_model[df_model["total_catch_tm"] > 0]
df_model["log_catch"] = np.log(df_model["total_catch_tm"])

y = df_model["log_catch"]
X = df_model[["hycom_temp_anom", "hycom_sal_anom_ref35"]]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res = mod.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


In [ ]:
df_model = df_merge.copy()
df_model = df_model[df_model["total_catch_tm"] > 0]
df_model = df_model[df_model["region"] != 3]

def resumir_3d(g):
    g = g.sort_values("date")
    num_cols = g.select_dtypes(include=[np.number]).columns
    return g.resample("3D", on="date")[num_cols].mean()

df_3d = (
    df_model
    .groupby("region", group_keys=True)
    .apply(resumir_3d)
    .dropna(subset=["total_catch_tm"])
)

df_3d = df_3d.reset_index().set_index(["region", "date"])
df_3d["log_catch"] = np.log(df_3d["total_catch_tm"])

y = df_3d["log_catch"]
X = df_3d[["hycom_temp_anom", "hycom_sal_anom_ref35"]]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res = mod.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


In [ ]:
df_model = df_merge.copy()
df_model = df_model.set_index(["dept", "date"])
df_model = df_model[df_model["total_catch_tm"] > 0]
df_model["log_catch"] = np.log(df_model["total_catch_tm"])

y = df_model["log_catch"]
X = df_model[["hycom_temp_anom", "hycom_sal_anom_ref35"]]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res = mod.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18,6))

def anio_pesquero(ts):
    return ts.year if ts.month >= 3 else ts.year - 1

df_plot = df_model.copy()
df_plot["anio_pesquero"] = df_plot["fecha"].apply(anio_pesquero)

base_cmap = plt.colormaps.get_cmap("tab20")

for region, dfg in df_plot.groupby("region_macro"):
    aps = sorted(dfg["anio_pesquero"].unique())
    colors = [base_cmap(i / max(1, len(aps) - 1)) for i in range(len(aps))]
    color_map = {ap: colors[i] for i, ap in enumerate(aps)}
    for ap, df in dfg.groupby("anio_pesquero"):
        df = df.sort_values("fecha")
        ax.plot(
            df["fecha"], df["total_pescado_tm"],
            lw=1.2, color=color_map[ap],
            marker="o", markersize=2,
            label=f"{region} {ap}"
        )

ax.set_xlabel("Fecha")
ax.set_ylabel("Toneladas totales")
ax.set_title("Pesca diaria total por región (colores por año pesquero marzo–febrero)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.6)
plt.legend(frameon=False, ncol=3, fontsize=8)
plt.tight_layout()
plt.show()
